In [34]:
# Libraries
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from sklearn.linear_model import Lasso, ElasticNet, LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from dml_simulation.AIPW.simulation_aipw import aipw_ate

In [35]:
# DPG 1
def dgp_regime1(n, rng, tau=1.0):
    p = 50
    X = rng.uniform(0, 1, size=(n, p))
    D = rng.binomial(1, 0.5, size=n)

    beta = np.zeros(p)
    s = int(0.05 * p)
    beta[:s] = 0.5
    
    Y = tau * D + X @ beta + rng.normal(0, 1, size=n)
    return X, D, Y


In [36]:
# DGP 2
from matplotlib.pylab import beta


def dgp_regime2(n, rng, tau=1.0):
    p = int(0.1 * n) # p grows with n but is still smaller than n
    X = rng.binomial(1, 0.5, size=(n, p))
    D = rng.binomial(1, 0.5, size=n)

    beta = np.zeros(p)
    s = int(0.05 * p)
    beta[:s] = 0.5
    
    Y = tau * D + X @ beta + rng.normal(0, 1, size=n)
    return X, D, Y

In [37]:
# DGP 3
def dgp_regime3(n, rng, tau=1.0):
    p = int(0.7 * n) # p grows with n and is smaller than n but by a smaller margin than in regime 2 
    X = rng.binomial(1, 0.5, size=(n, p))
    D = rng.binomial(1, 0.5, size=n)

    beta = np.zeros(p)
    s = int(0.05 * p)
    beta[:s] = 0.5

    Y = tau * D + X @ beta + rng.normal(0, 1, size=n)
    return X, D, Y

In [38]:
# DGP 4
def dgp_regime4(n, rng, tau=1.0):
    p = int(1.5 * n) # p grows with n and is much larger than n but still sparse
    X = rng.binomial(1, 0.5, size=(n, p))
    D = rng.binomial(1, 0.5, size=n)

    beta = np.zeros(p)
    s = int(0.05 * p)
    beta[:s] = 0.5

    Y = tau * D + X @ beta + rng.normal(0, 1, size=n)
    return X, D, Y

In [39]:
# Learners
learners_regime = {
    "OLS": LinearRegression(),
    "Lasso": Lasso(alpha=0.05),
    "ElasticNet": ElasticNet(alpha=0.1, l1_ratio=0.5),
    "RF": RandomForestRegressor(n_estimators=100, n_jobs=1),
    "GB": GradientBoostingRegressor(n_estimators=100)
}

In [40]:
# Monte Carlo Simulation
def run_simulation(dgp_func, learners, n, tau_true, seed):
    rng = np.random.default_rng(seed)
    X, D, Y = dgp_func(n, rng, tau=tau_true)

    estimates = {}

    for name, model in learners.items():

        # Propensity
        prop = LogisticRegression(max_iter=1000)
        prop.fit(X, D)
        e_hat = prop.predict_proba(X)[:, 1]

        # Outcome models
        model.fit(X[D==1], Y[D==1])
        m1_hat = model.predict(X)

        model.fit(X[D==0], Y[D==0])
        m0_hat = model.predict(X)

        tau_hat = aipw_ate(Y, D, e_hat, m0_hat, m1_hat)

        estimates[name] = tau_hat

    return estimates


def monte_carlo(dgp_func, learners, n, tau_true=1.0, sims=100):
    results = Parallel(n_jobs=-1)(
        delayed(run_simulation)(dgp_func, learners, n, tau_true, i)
        for i in range(sims)
    )

    df = pd.DataFrame(results)

    summary = pd.DataFrame({
        "Mean": df.mean(),
        "Bias": df.mean() - tau_true,
        "Variance": df.var(),
        "RMSE": np.sqrt(df.var() + (df.mean() - tau_true)**2)
    })

    return summary

In [41]:
print("Regime 1")
print(monte_carlo(dgp_regime1, learners_regime, n=100))

print("Regime 2")
print(monte_carlo(dgp_regime2, learners_regime, n=100))

print("Regime 3")
print(monte_carlo(dgp_regime3, learners_regime, n=100))

print("Regime 4")
print(monte_carlo(dgp_regime4, learners_regime, n=100))

Regime 1


                Mean      Bias  Variance      RMSE
OLS         1.050910  0.050910  1.188698  1.091462
Lasso       1.011046  0.011046  0.033330  0.182899
ElasticNet  1.012922  0.012922  0.032644  0.181139
RF          1.011849  0.011849  0.034089  0.185012
GB          1.011437  0.011437  0.036473  0.191321
Regime 2
                Mean      Bias  Variance      RMSE
OLS         1.039180  0.039180  0.038812  0.200866
Lasso       1.034053  0.034053  0.036699  0.194574
ElasticNet  1.034071  0.034071  0.036626  0.194389
RF          1.030579  0.030579  0.039687  0.201549
GB          1.034261  0.034261  0.052800  0.232323
Regime 3
                Mean      Bias  Variance      RMSE
OLS         1.004580  0.004580  0.083973  0.289818
Lasso       1.003500  0.003500  0.036972  0.192313
ElasticNet  1.002775  0.002775  0.036540  0.191174
RF          1.010190  0.010190  0.034258  0.185369
GB          1.014944  0.014944  0.038671  0.197215
Regime 4
                Mean      Bias  Variance      RMSE
OLS 